In [ ]:
# This notebook demonstrates the updated albedo handling

import os
import sys
import logging
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

# Set up logging
logging.basicConfig(level=logging.DEBUG)

# Import PyRadtran modules
from pyradtran.config import PathsConfig, SimulationDefaults, SimulationConfig, ExecutionConfig, OutputConfig, load_config
from pyradtran.core import Simulation
from pyradtran.interface import PyRadtranAccessor  # This should register the accessor automatically

# Define paths
LIBRADTRAN_DATA_PATH = '/opt/libradtran/2.0.4/share/libRadtran/data'
LIBRADTRAN_EXEC_PATH = '/opt/libradtran/2.0.4/bin/uvspec'
ATMOSPHERE_FILE = '/projekt_agmwend/data/HALO-AC3/05_VELOX_Tools/add_data/afglsw.dat'
SOLAR_SPECTRUM_FILE = '/projekt_agmwend/home_rad/sophie/libradtran/solar_flux/NewGuey2003.dat'
WORKING_DIR = os.path.join('/projekt_agmwend/home_rad/Joshua/HALO-AC3_Arctic_leads/sim/pyradtran', 'work')

# Create working directory if it doesn't exist
os.makedirs(WORKING_DIR, exist_ok=True)

# Create a test dataset with varying albedo values
def create_test_dataset():
    # Create a time series
    times = pd.date_range(start="2022-04-01", periods=3, freq="1H")
    
    # Create location and albedo data
    latitudes = np.array([75.0, 75.0, 75.0])
    longitudes = np.array([0.0, 0.0, 0.0])
    
    # Different albedo for each timestep
    albedo_values = np.array([0.3, 0.5, 0.8])
    
    # Create the dataset
    ds = xr.Dataset(
        coords={
            'time': times,
            'altitude': [0.0]  # Single altitude level
        },
        data_vars={
            'latitude': ('time', latitudes),
            'longitude': ('time', longitudes),
            'albedo': ('time', albedo_values)  # Add albedo as a data variable with time dimension
        }
    )
    
    return ds

# Create test config
def create_test_config():
    config = SimulationConfig(
        paths=PathsConfig(
            libradtran_bin=LIBRADTRAN_EXEC_PATH,
            libradtran_data=LIBRADTRAN_DATA_PATH,
            atmosphere_profile=ATMOSPHERE_FILE,
            solar_spectrum=SOLAR_SPECTRUM_FILE,
            output_dir=WORKING_DIR,
            working_dir=WORKING_DIR
        ),
        simulation_defaults=SimulationDefaults(
            rte_solver='disort',
            mol_abs_param='lowtran per_nm',
            wavelength_nm=[400, 700],
            output_columns=['sza', 'edir', 'eglo', 'edn', 'eup', 'enet', 'albedo'],
            output_altitudes_km=[0.0],
            albedo_type='const',
            albedo_value=0.1,  # This should be overridden by the dataset values
            aerosols={'enabled': False},
            clouds={'enabled': False},
            integrate_wavelength=True,
            additional_options=["output_process per_nm", "output_process integrate"]
        ),
        execution=ExecutionConfig(
            max_workers=1,
            cleanup_temp_files=False,
            debug_mode=True,
            timeout_seconds=60
        ),
        output=OutputConfig(
            filename_prefix="albedo_test",
            filename_suffix=".nc",
            netcdf_encoding=None
        )
    )
    
    return config

# Create test dataset and config
test_ds = create_test_dataset()
test_config = create_test_config()

print("Test Dataset:")
print(test_ds)

# Save config to file
import yaml
from dataclasses import asdict
from pathlib import Path

# Convert config to dictionary
config_dict = {
    'paths': asdict(test_config.paths),
    'simulation_defaults': asdict(test_config.simulation_defaults),
    'execution': asdict(test_config.execution),
    'output': asdict(test_config.output)
}

# Convert Path objects to strings
def convert_paths_to_strings(d):
    if isinstance(d, dict):
        return {k: convert_paths_to_strings(v) for k, v in d.items()}
    elif isinstance(d, list):
        return [convert_paths_to_strings(v) for v in d]
    elif isinstance(d, Path):
        return str(d)
    else:
        return d

config_dict = convert_paths_to_strings(config_dict)

# Save configuration to YAML file
config_file_path = os.path.join(WORKING_DIR, 'albedo_test_config.yaml')
with open(config_file_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False)
    
print(f"Config saved to {config_file_path}")

# Run the simulation
print("Running simulation with varying albedo values...")
result_ds = test_ds.pyradtran.run_uvspec(
    config_path=config_file_path,
    return_dataset=True,
    save_to_file=True
)

print("\nSimulation result:")
print(result_ds)

# Check input files to verify albedo values
import glob
input_files = glob.glob(os.path.join(WORKING_DIR, "uvspec_*.inp"))

print("\nChecking input files for albedo values:")
for input_file in input_files:
    with open(input_file, 'r') as f:
        content = f.read()
        # Extract the albedo line
        albedo_line = [line for line in content.split('\n') if line.startswith('albedo')][0]
        print(f"{os.path.basename(input_file)}: {albedo_line}")